In [1]:
gold_path = "src/data/conllu/UD_Russian-SynTagRus/ru_syntagrus-ud-test.conllu" # TODO

In [2]:
from src.score.score_functions import create_edges

In [3]:
def delete_point_nodes(sent_dict):
    return [t for t in sent_dict if "." not in t["id"]]

In [4]:
from collections import Counter
from copy import deepcopy

def create_statistics(gold_tree1_conll, output_tree1):
    gold_tree1 = [{'id': str(t['id']), 'form': t['form'],
               'parent_id': str(t['head']), 'relation': t['deprel'],
               'pos': t['upos'], 'feats': t['feats']} for t in gold_tree1_conll]
    gold_tree1 = delete_point_nodes(gold_tree1)

    res = {}
    res["gold_tree"] = deepcopy(gold_tree1)
    res["sent_text"] = gold_tree1_conll.metadata["text"]
    res["sent_id"] = gold_tree1_conll.metadata["sent_id"]
    res["pred_tree"] = output_tree1["pred_tree"]

    if isinstance(output_tree1["pred_tree"], list):
        
        output_tree1["pred_tree"] = delete_point_nodes(output_tree1["pred_tree"])

        pred_unlabeled_edges, pred_labeled_edges, pred_unlabeled_edges_set, pred_labeled_edges_set = create_edges(output_tree1["pred_tree"])
        pred_nodes_set = { r[0] for r in pred_labeled_edges }
        #print(pred_nodes_set)
        assert len(output_tree1["pred_tree"]) == len(pred_unlabeled_edges)
        assert len(output_tree1["pred_tree"]) == len(pred_labeled_edges)

        gold_unlabeled_edges, gold_labeled_edges, _, _ = create_edges(res["gold_tree"])
        assert len(res["gold_tree"]) == len(gold_unlabeled_edges)
        assert len(res["gold_tree"]) == len(gold_labeled_edges)
    
        for t_i in range(len(res["gold_tree"])):
            assert res["gold_tree"][t_i]["form"] in gold_unlabeled_edges[t_i][0]
            assert res["gold_tree"][t_i]["form"] in gold_labeled_edges[t_i][0]
            res["gold_tree"][t_i]["unlab_edge"] = gold_unlabeled_edges[t_i]
            res["gold_tree"][t_i]["lab_edge"] = gold_labeled_edges[t_i]
            if res["gold_tree"][t_i]["unlab_edge"][0] not in pred_nodes_set:
                category = 1
            elif res["gold_tree"][t_i]["unlab_edge"] not in pred_unlabeled_edges_set:
                category = 2
            elif res["gold_tree"][t_i]["lab_edge"] not in pred_labeled_edges_set:
                category = 3
            else:
                category = 4
            res["gold_tree"][t_i]["category"] = category

        res["categories"] = Counter(t["category"] for t in res["gold_tree"])
 
        res["gold_len"] = len(res["gold_tree"])
        res["pred_len"] = len(pred_unlabeled_edges)
    else:
        res["gold_len"], res["pred_len"] = None, None
        res["categories"] = None
    return res

In [5]:
import pandas as pd

In [6]:
def create_experiment_results(sentences, pred_trees):
    assert len(sentences) == len(pred_trees)
    
    results = []
    for sent_i in range(len(sentences)):
        output_r = create_statistics(sentences[sent_i], pred_trees[sent_i])
        results.append(output_r)
    return results

In [7]:
def calculate_metrics(sent_res):
    if sent_res["categories"] is not None:
        uas_precision = (sent_res["categories"][3] + sent_res["categories"][4]) / sent_res["pred_len"]
        uas_recall = (sent_res["categories"][3] + sent_res["categories"][4]) / sent_res["gold_len"]
        if uas_precision + uas_recall > 0:
            uas = (2 * uas_precision * uas_recall) / (uas_precision + uas_recall)
        else:
            uas = 0.0
        las_precision = sent_res["categories"][4] / sent_res["pred_len"]
        las_recall = sent_res["categories"][4] / sent_res["gold_len"]
        if (las_precision + las_recall) > 0:
            las = (2 * las_precision * las_recall) / (las_precision + las_recall)
        else:
            las = 0.0
        return uas, las
    else:
        return None, None

In [8]:
def create_experiment_metrics(results):
    uas_metrics, las_metrics = [], []
    for r in results:
        sent_uas, sent_las = calculate_metrics(r)
        uas_metrics.append(sent_uas)
        las_metrics.append(sent_las)
    return uas_metrics, las_metrics

In [9]:
import yaml

with open('result_paths.yaml', 'r') as file:
    configs = yaml.safe_load(file)

In [10]:
configs

[{'model_name': 'Qwen06_Base',
  'representation_type': 'grct',
  'pred_result_path': 'pred_results/Qwen06_Base_grct_syntagrus.json',
  'metric_path': 'metrics/metrics_Qwen06_Base_grct_syntagrus.json'},
 {'model_name': 'Qwen06_Base',
  'representation_type': 'lct',
  'pred_result_path': 'pred_results/Qwen06_Base_lct_syntagrus.json',
  'metric_path': 'metrics/metrics_Qwen06_Base_lct_syntagrus.json'},
 {'model_name': 'Qwen06_Instruct',
  'representation_type': 'grct',
  'pred_result_path': 'pred_results/Qwen06_Instruct_grct_syntagrus.json',
  'metric_path': 'metrics/metrics_Qwen06_Instruct_grct_syntagrus.json'},
 {'model_name': 'Qwen06_Instruct',
  'representation_type': 'lct',
  'pred_result_path': 'pred_results/Qwen06_Instruct_lct_syntagrus.json',
  'metric_path': 'metrics/metrics_Qwen06_Instruct_lct_syntagrus.json'},
 {'model_name': 'Qwen4_Base',
  'representation_type': 'grct',
  'pred_result_path': 'pred_results/Qwen4_Base_grct_syntagrus.json',
  'metric_path': 'metrics/metrics_Qwen

In [11]:
# pred_path = "pred_results/Qwen06_Base_grct_syntagrus.json"

In [12]:
from conllu import parse_tree, parse
import json

with open(gold_path, 'r') as file:
    content = file.read()
sentences = parse(content)

config_results = {}
config_uas, config_las = {}, {}
for config in configs:
    config_name = f"{config['model_name']}_{config['representation_type']}"
    print(config_name)
    with open(config["pred_result_path"], 'r') as f:
        pred_trees = json.load(f)
    config_results[config_name] = create_experiment_results(sentences, pred_trees)
    config_uas[config_name], config_las[config_name] = create_experiment_metrics(config_results[config_name])

Qwen06_Base_grct
Qwen06_Base_lct
Qwen06_Instruct_grct
Qwen06_Instruct_lct
Qwen4_Base_grct
Qwen4_Base_lct
Qwen4_Instruct_grct
Qwen4_Instruct_lct
Ruadapt4_Hybrid_grct
Ruadapt4_Hybrid_lct
Qwen8_Base_grct
Qwen8_Base_lct
Qwen8_Instruct_grct
Qwen8_Instruct_lct
Ruadapt8_Instruct_grct
Ruadapt8_Instruct_lct
Llama7_Base_grct
Qwen06_Base_lr5_lct


In [13]:
def print_mean_metrics(uas_metrics, las_metrics):
    good_uas = [r for r in uas_metrics if r is not None]
    bad_uas = [r for r in uas_metrics if r is None]
    good_las = [r for r in las_metrics if r is not None]
    bad_las = [r for r in las_metrics if r is None]
    if good_uas:
        print(f"{sum(good_uas) / len(good_uas) * 100:.1f}% ({sum(good_uas) / len(uas_metrics) * 100:.1f})%")
    if good_las:
        print(f"{sum(good_las) / len(good_las) * 100:.1f}% ({sum(good_las) / len(las_metrics) * 100:.1f})%")
    print(len(bad_uas), len(bad_las))

In [14]:
for config_name in config_uas:
    print(config_name)
    print_mean_metrics(config_uas[config_name], config_las[config_name])
    print()

Qwen06_Base_grct
93.8% (54.2)%
91.6% (53.0)%
3711 3711

Qwen06_Base_lct
8800 8800

Qwen06_Instruct_grct
93.9% (93.8)%
91.7% (91.5)%
10 10

Qwen06_Instruct_lct
8800 8800

Qwen4_Base_grct
93.5% (42.3)%
91.0% (41.1)%
4821 4821

Qwen4_Base_lct
93.6% (85.0)%
91.7% (83.2)%
811 811

Qwen4_Instruct_grct
93.6% (93.4)%
91.5% (91.4)%
17 17

Qwen4_Instruct_lct
93.1% (92.9)%
91.2% (91.0)%
17 17

Ruadapt4_Hybrid_grct
94.9% (94.9)%
93.0% (92.9)%
6 6

Ruadapt4_Hybrid_lct
93.4% (93.3)%
91.4% (91.3)%
7 7

Qwen8_Base_grct
95.2% (95.2)%
93.3% (93.2)%
2 2

Qwen8_Base_lct
95.1% (95.0)%
93.2% (93.1)%
8 8

Qwen8_Instruct_grct
95.2% (95.1)%
93.3% (93.2)%
9 9

Qwen8_Instruct_lct
95.2% (95.1)%
93.3% (93.2)%
7 7

Ruadapt8_Instruct_grct
94.7% (94.7)%
92.8% (92.8)%
1 1

Ruadapt8_Instruct_lct
94.2% (94.2)%
92.3% (92.2)%
2 2

Llama7_Base_grct
93.2% (93.0)%
87.3% (87.1)%
19 19

Qwen06_Base_lr5_lct
57.1% (51.6)%
49.6% (44.7)%
858 858

